In [1]:
%%capture
!pip install transformers[torch] datasets evaluate rouge_score nltk numpy pandas

In [ ]:

import os
import pandas as pd
import glob
import torch
import evaluate
import numpy as np
import nltk
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer, 
    DataCollatorForSeq2Seq
)
from datasets import Dataset
from tqdm.auto import tqdm

# 1. INSTALL MISSING LIBRARIES
os.system("pip install -q evaluate rouge_score")
# We save to the 'working' directory so you can download the model/CSV later
save_path = "/kaggle/working/bart_final_model"
print(f" Outputs will be saved to: {save_path}")

# 3. ROBUST DATA LOADING
print("Searching for files in /kaggle/input/...")
def get_path(filename):
    # Kaggle keeps data in ../input or /kaggle/input
    found = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    return found[0] if found else None

# Load Train
train_files = ["train.csv", "train_2.csv", "train_3.csv"]
dfs = []
for name in train_files:
    p = get_path(name)
    if p: 
        print(f"   Found: {name}")
        dfs.append(pd.read_csv(p))
    else:
        print(f" WARNING: Could not find {name}")

if not dfs:
    raise ValueError("No training data found! Check your dataset mapping.")

full_train_df = pd.concat(dfs, ignore_index=True)
full_train_df.dropna(subset=['dialogue', 'summary'], inplace=True)
full_train_df = full_train_df.drop_duplicates(subset=['dialogue'])
full_train_df['dialogue'] = full_train_df['dialogue'].astype(str)
full_train_df['summary'] = full_train_df['summary'].astype(str)

# Load Test
test_path = get_path("test_2.csv")
if not test_path:
    raise ValueError("Could not find test_2.csv!")
    
df_test = pd.read_csv(test_path)
df_test['dialogue'] = df_test['dialogue'].astype(str)

print(f" Data Loaded: {len(full_train_df)} Training rows, {len(df_test)} Test rows.")

# 4. PREPARE MODEL
model_id = "linydub/bart-large-samsum"
print(f"Loading {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# 5. TOKENIZATION
def preprocess_function(examples):
    inputs = [doc for doc in examples["dialogue"]]
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True)
    labels = tokenizer(text_target=examples["summary"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Convert to HF Dataset
full_dataset = Dataset.from_pandas(full_train_df)
train_val = full_dataset.train_test_split(test_size=0.01, seed=42) 
tokenized = train_val.map(preprocess_function, batched=True)

# 6. METRICS
rouge = evaluate.load("rouge")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    decoded_preds = ["\n".join(nltk.sent_tokenize(p.strip())) for p in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(l.strip())) for l in decoded_labels]
    
    return rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

# 7. TRAINING CONFIG (The 0.50 Strategy)
args = Seq2SeqTrainingArguments(
    output_dir=save_path,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_steps=100,
    predict_with_generate=True,
    fp16=True,
    report_to="none",
    save_total_limit=1
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics
)

print(" STARTING KAGGLE TRAINING...")
trainer.train()
trainer.save_model(save_path)

# 8. GENERATION (Config-Aligned Strategy: Beams=6)
print(" Generating FINAL SUBMISSION...")
model.eval()
batch_size = 8
inputs = df_test["dialogue"].tolist()
predictions = []

for i in tqdm(range(0, len(inputs), batch_size)):
    batch = inputs[i : i + batch_size]
    model_inputs = tokenizer(batch, max_length=1024, truncation=True, padding=True, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **model_inputs, 
            max_new_tokens=140,      
            min_length=20,           
            num_beams=6,             # Aligned with model config (Native=6)
            length_penalty=2.0,      # Proven best setting
            no_repeat_ngram_size=3,  # Clean output
            early_stopping=True
        )
    predictions.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))

# 9. SAVE SUBMISSION
final_summaries = [p.replace("\n", " ").strip() for p in predictions]
submission_df = pd.DataFrame({"id": df_test["id"], "summary": final_summaries})
submission_df.to_csv("submission.csv", index=False)
print(" DONE! submission.csv created in /kaggle/working/")

✅ Outputs will be saved to: /kaggle/working/bart_final_model
🔍 Searching for files in /kaggle/input/...
   Found: train.csv
   Found: train_2.csv
   Found: train_3.csv
📊 Data Loaded: 1973 Training rows, 500 Test rows.
🔄 Loading linydub/bart-large-samsum...


Map:   0%|          | 0/1953 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

/tmp/ipykernel_55/4271982614.py:120: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


🔥 STARTING KAGGLE TRAINING...


...

Step,Training Loss,Validation Loss


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 62, 'min_length': 11, 'early_stopping': True, 'num_beams': 6, 'no_repeat_ngram_size': 3}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


🚀 Generating FINAL SUBMISSION...


  0%|          | 0/63 [00:00<?, ?it/s]

✅ DONE! submission.csv created in /kaggle/working/
